# Préparation des données (Preprocessing)
Ce notebook détaille les étapes de nettoyage, d'imputation et de préparation des données pour la modélisation de notre projet 'Eco-Smart Classifier'.


In [1]:
import pandas as pd
import numpy as np
import os, warnings
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

NUMERIC = ['Poids', 'Volume', 'Conductivite', 'Opacite', 'Rigidite', 'Prix_Revente']
TARGET = 'Categorie'
SOURCE_COL = 'Source' 

## Étape 0 : Chargement des données
Chargement du jeu de données brut depuis le dossier `data/raw/` et première exploration (taille, types, valeurs manquantes).


In [2]:
df = pd.read_csv("../data/raw/dataset_ProjetML_2026.csv")
print(f"Shape initiale : {df.shape}\n")
print("Types de colonnes :")
print(df.dtypes)
print("\nAperçu (5 premières lignes) :")
display(df.head())
print("\nStatistiques descriptives (numériques) :")
display(df[NUMERIC].describe().round(3))

Shape initiale : (10500, 9)

Types de colonnes :
Poids               float64
Volume              float64
Conductivite        float64
Opacite             float64
Rigidite            float64
Prix_Revente        float64
Categorie            object
Source               object
Rapport_Collecte     object
dtype: object

Aperçu (5 premières lignes) :


,Poids,Volume,Conductivite,Opacite,Rigidite,Prix_Revente,Categorie,Source,Rapport_Collecte
0,16.708780,70.940977,0.0,1.0,1.0,0.835439,Papier,NaN,Lot de papier récupéré dans un site non rensei...
1,47.277476,64.702925,0.0,NaN,3.0,4.727748,Plastique,Usine_A,"Lot plastique à l'Usine A. Volume 64.7 L, poid..."
2,NaN,317.415183,0.0,NaN,9.0,4.211790,Verre,Usine_B,Bris de verre ou contenants en provenance de l...
3,NaN,21.474391,0.0,NaN,1.0,0.442067,Papier,Centre_Tri,Feuilles et cartons collectés au Centre de Tri...
4,NaN,59.462176,0.0,1.0,NaN,0.723004,Papier,Usine_B,Déchet de type papier identifié à l'Usine B. V...



Statistiques descriptives (numériques) :


,Poids,Volume,Conductivite,Opacite,Rigidite,Prix_Revente
count,9471.000,9960.000,9483.000,9465.000,9942.000,9964.000
mean,77.797,144.408,0.208,1.160,5.887,58.588
std,127.847,136.384,0.379,5.493,3.087,720.059
min,-99.000,-26.808,0.000,0.000,1.000,-50.000
25%,19.752,44.437,0.000,0.196,3.000,1.394
50%,39.193,88.084,0.000,0.553,5.000,4.135
75%,130.498,240.200,0.000,1.000,9.000,6.782
max,2334.219,554.107,0.999,55.000,10.000,9999.000


## Étape 1 : Analyse MCAR / MAR / MNAR
Nous allons évaluer la nature des valeurs manquantes. Un test de Little nous permet de déterminer si les données manquantes sont de type MCAR (Missing Completely At Random).


In [3]:
def littles_mcar_test(data: pd.DataFrame) -> dict:
    """
    Implémentation simplifiée du test de Little (1988).
    Retourne la statistique chi2, les degrés de liberté et la p-value.
    H0 : les données sont MCAR.
    Si p-value > 0.05 → on ne rejette pas MCAR.
    """
    df_num = data.copy()
    n, p = df_num.shape

    R = df_num.isnull().astype(int)
    patterns = R.drop_duplicates()
    grand_mean = df_num.mean()
    grand_cov  = df_num.cov()

    chi2_stat = 0.0
    df_freedom = 0

    for _, pattern in patterns.iterrows():
        observed_cols = pattern[pattern == 0].index.tolist()
        if not observed_cols:
            continue

        mask = (R == pattern.values).all(axis=1)
        subset = df_num.loc[mask, observed_cols]
        n_k = len(subset)

        if n_k < 2:
            continue

        mean_k = subset.mean()
        diff   = (mean_k - grand_mean[observed_cols]).values

        try:
            cov_k = grand_cov.loc[observed_cols, observed_cols].values
            cov_inv = np.linalg.pinv(cov_k)
            chi2_stat += n_k * diff @ cov_inv @ diff
            df_freedom += len(observed_cols)
        except Exception:
            continue

    p_value = 1 - stats.chi2.cdf(chi2_stat, df=max(df_freedom - p, 1))
    return {"chi2": round(chi2_stat, 4), "df": df_freedom, "p_value": round(p_value, 4)}

mcar_result = littles_mcar_test(df[NUMERIC])
print(f"Little's MCAR Test (variables numériques) :")
print(f"  chi² = {mcar_result['chi2']},  ddl = {mcar_result['df']},  p = {mcar_result['p_value']}")
if mcar_result['p_value'] > 0.05:
    print("  → H0 non rejetée : les manquants sont probablement MCAR")
else:
    print("  → H0 rejetée : les manquants ne sont PAS MCAR (MAR ou MNAR probable)")

Little's MCAR Test (variables numériques) :
  chi² = 184.2081,  ddl = 144,  p = 0.0052
  → H0 rejetée : les manquants ne sont PAS MCAR (MAR ou MNAR probable)


In [4]:
missing_matrix = df[NUMERIC].isnull().astype(int)
fig_missing = px.imshow(
    missing_matrix.T,
    color_continuous_scale=["#2ecc71", "#e74c3c"],
    title="Carte des valeurs manquantes (rouge = NaN)",
    labels={"x": "Observations", "y": "Variable", "color": "Manquant"},
    aspect="auto",
    height=350
)
fig_missing.update_layout(coloraxis_showscale=False)
fig_missing.show()

In [5]:
missing_pct = df[NUMERIC].isnull().mean() * 100
fig_bar = px.bar(
    x=missing_pct.index,
    y=missing_pct.values,
    labels={"x": "Variable", "y": "% Manquants"},
    title="Taux de valeurs manquantes par variable numérique",
    color=missing_pct.values,
    color_continuous_scale="RdYlGn_r",
    height=400
)
fig_bar.update_layout(coloraxis_showscale=False)
fig_bar.show()

## Étape 2 : Traitement des Outliers
Nous appliquons des règles métiers pour corriger ou supprimer les valeurs aberrantes. Celles-ci sont transformées en NaN afin de les imputer lors de la prochaine étape.


In [6]:
df_clean = df.copy()

# Poids : -99 indique une valeur inconnue
df_clean.loc[df_clean['Poids'] == -99, 'Poids'] = np.nan

# Volume : erreurs de signe corrigées avec la valeur absolue
df_clean.loc[df_clean['Volume'] < 0, 'Volume'] = df_clean['Volume'].abs()

# Opacite : échelle de 0 à 1 (55.0 est une erreur)
df_clean.loc[df_clean['Opacite'] == 55, 'Opacite'] = np.nan

# Prix_Revente : 9999 ou -50 sont des codes erreurs/manquants
df_clean.loc[df_clean['Prix_Revente'].isin([9999, -50]), 'Prix_Revente'] = np.nan

print("Outliers convertis en NaN. Valeurs manquantes totales par colonne :")
print(df_clean[NUMERIC].isnull().sum())

Outliers convertis en NaN. Valeurs manquantes totales par colonne :
Poids           1133
Volume           540
Conductivite    1017
Opacite         1132
Rigidite         558
Prix_Revente     598
dtype: int64


In [7]:
fig_box = make_subplots(rows=2, cols=3, subplot_titles=NUMERIC)
for i, col in enumerate(NUMERIC):
    r, c = divmod(i, 3)
    fig_box.add_trace(
        go.Box(y=df[col].dropna(), name="Avant", marker_color="#e74c3c", showlegend=(i == 0)),
        row=r+1, col=c+1
    )
    fig_box.add_trace(
        go.Box(y=df_clean[col].dropna(), name="Après", marker_color="#2ecc71", showlegend=(i == 0)),
        row=r+1, col=c+1
    )
fig_box.update_layout(title_text="Boxplots avant / après traitement des outliers", height=600)
fig_box.show()

## Étape 3 : Comparaison des 3 imputeurs
Afin d'obtenir l'imputation la plus fidèle, nous comparons les performances (RMSE) de trois algorithmes : la Médiane, KNN, et IterativeImputer.


In [8]:
df_complet = df_clean[NUMERIC].dropna().copy()
np.random.seed(42)
nb_test   = int(len(df_complet) * 0.10)
idx_test  = np.random.choice(df_complet.index, size=nb_test, replace=False)

vraies_valeurs  = df_complet.loc[idx_test].copy()
df_pour_imputer = df_complet.copy()
df_pour_imputer.loc[idx_test] = np.nan

# Médiane
imp_med  = SimpleImputer(strategy='median')
arr_med  = imp_med.fit_transform(df_pour_imputer)
pred_med = pd.DataFrame(arr_med, index=df_complet.index, columns=NUMERIC).loc[idx_test]
rmse_med = np.sqrt(mean_squared_error(vraies_valeurs, pred_med))

# KNN
imp_knn  = KNNImputer(n_neighbors=5)
arr_knn  = imp_knn.fit_transform(df_pour_imputer)
pred_knn = pd.DataFrame(arr_knn, index=df_complet.index, columns=NUMERIC).loc[idx_test]
rmse_knn = np.sqrt(mean_squared_error(vraies_valeurs, pred_knn))

# IterativeImputer
imp_iter  = IterativeImputer(max_iter=10, random_state=42)
arr_iter  = imp_iter.fit_transform(df_pour_imputer)
pred_iter = pd.DataFrame(arr_iter, index=df_complet.index, columns=NUMERIC).loc[idx_test]
rmse_iter = np.sqrt(mean_squared_error(vraies_valeurs, pred_iter))

rmse_dict = {"Médiane": rmse_med, "KNN (k=5)": rmse_knn, "IterativeImputer": rmse_iter}
for nom, rmse in rmse_dict.items():
    print(f"{nom:<22} → RMSE = {round(rmse, 4)}")

Médiane                → RMSE = 90.7123
KNN (k=5)              → RMSE = 85.1407
IterativeImputer       → RMSE = 85.1407


In [9]:
fig_rmse = px.bar(
    x=list(rmse_dict.keys()),
    y=list(rmse_dict.values()),
    color=list(rmse_dict.keys()),
    title="Comparaison des imputeurs (RMSE — plus petit = meilleur)",
    labels={"x": "Imputeur", "y": "RMSE"},
    color_discrete_sequence=["#e74c3c", "#3498db", "#2ecc71"],
    height=400
)
fig_rmse.show()

In [10]:
# Application d'un imputeur test pour la visualisation
imp_test = IterativeImputer(max_iter=10, random_state=42)
arr_test = imp_test.fit_transform(df_clean[NUMERIC])
df_imputed_test = pd.DataFrame(arr_test, columns=NUMERIC, index=df_clean.index)

fig_dist_imp = make_subplots(rows=2, cols=3, subplot_titles=NUMERIC)
for i, col in enumerate(NUMERIC):
    r, c = divmod(i, 3)
    fig_dist_imp.add_trace(
        go.Histogram(x=df_clean[col], name="Avant (avec NaN)", marker_color="#3498db", opacity=0.7, showlegend=(i == 0)),
        row=r+1, col=c+1
    )
    fig_dist_imp.add_trace(
        go.Histogram(x=df_imputed_test[col], name="Après", marker_color="#e67e22", opacity=0.7, showlegend=(i == 0)),
        row=r+1, col=c+1
    )
fig_dist_imp.update_layout(barmode='overlay', title_text="Distributions avant / après imputation", height=600)
fig_dist_imp.show()

## Étape 4 : Imputation finale
L'IterativeImputer offre les meilleures performances RMSE, nous l'utilisons pour l'ensemble des données manquantes. Nous appliquons également des contraintes métier sur les données générées (ex. pas de poids négatifs).


In [11]:
imp_final = IterativeImputer(max_iter=10, random_state=42)
arr_final = imp_final.fit_transform(df_clean[NUMERIC])
df_imputed = pd.DataFrame(arr_final, columns=NUMERIC, index=df_clean.index)

# Contraintes métier post-imputation
df_imputed['Rigidite'] = df_imputed['Rigidite'].clip(1, 10)
df_imputed['Opacite']  = df_imputed['Opacite'].clip(0, 1)
df_imputed['Poids']    = df_imputed['Poids'].clip(lower=0)
df_imputed['Volume']   = df_imputed['Volume'].clip(lower=0)

print(f"NaN restants après imputation : {df_imputed.isnull().sum().sum()}")

NaN restants après imputation : 0


## Étape 5 : Standardisation
Nous standardisons les variables (StandardScaler) afin que les modèles ultérieurs ne soient pas biaisés par les différences d'échelle entre variables.


In [12]:
scaler = StandardScaler()
df_scaled = df_imputed.copy()
df_scaled[NUMERIC] = scaler.fit_transform(df_scaled[NUMERIC])

print("Statistiques descriptives après standardisation (moyenne ≈ 0, écart-type ≈ 1) :")
display(df_scaled[NUMERIC].describe().round(3))

Statistiques descriptives après standardisation (moyenne ≈ 0, écart-type ≈ 1) :


,Poids,Volume,Conductivite,Opacite,Rigidite,Prix_Revente
count,10500.000,10500.000,10500.000,10500.000,10500.000,10500.000
mean,0.000,0.000,-0.000,-0.000,0.000,0.000
std,1.000,1.000,1.000,1.000,1.000,1.000
min,-0.647,-1.073,-1.255,-1.555,-1.587,-1.313
25%,-0.480,-0.741,-0.546,-1.044,-0.938,-0.732
50%,-0.316,-0.415,-0.546,-0.079,-0.289,-0.347
75%,0.431,0.705,-0.365,1.041,1.010,0.039
max,18.416,3.942,2.797,1.041,1.335,3.313


## Étape 6 : Encodage Source
La source de la donnée étant une variable nominale (catégories sans ordre inhérent), nous utilisons un encodage One-Hot.


In [13]:
source_series = df_clean[SOURCE_COL].fillna('Inconnu')
source_dummies = pd.get_dummies(
    source_series,
    prefix='Source',
    drop_first=False,
    dtype=int
)

le_source = LabelEncoder()
source_label_enc = le_source.fit_transform(source_series)

df_final = df_scaled.copy()
df_final[source_dummies.columns] = source_dummies.values
df_final['Source_encoded'] = source_label_enc
df_final[TARGET] = df_clean[TARGET].values

if 'Rapport_Collecte' in df_clean.columns:
    df_final['Rapport_Collecte'] = df_clean['Rapport_Collecte'].values

print(f"Shape après encodage : {df_final.shape}")

Shape après encodage : (10500, 14)


In [14]:
print("Colonnes OHE créées :", source_dummies.columns.tolist())
display(df_final[['Source_encoded'] + source_dummies.columns.tolist()].head())

Colonnes OHE créées : ['Source_Centre_Tri', 'Source_Collecte_Citoyenne', 'Source_Inconnu', 'Source_Usine_A', 'Source_Usine_B']


,Source_encoded,Source_Centre_Tri,Source_Collecte_Citoyenne,Source_Inconnu,Source_Usine_A,Source_Usine_B
0,2,0,0,1,0,0
1,3,0,0,0,1,0
2,4,0,0,0,0,1
3,0,1,0,0,0,0
4,4,0,0,0,0,1


## Étape 7 : Split 70:15:15
Les données sont scindées en jeux d'entraînement, de validation et de test tout en respectant les proportions de la cible `Categorie` via la stratification.


In [18]:
# ── Étape 7 : Split 70 / 15 / 15 ────────────────────────────────────────────
df_labeled   = df_final[df_final[TARGET].notna()].copy()
df_unlabeled = df_final[df_final[TARGET].isna()].copy()

# Toutes les colonnes sauf la cible (inclut Rapport_Collecte et Prix_Revente)
FEATURE_COLS = [c for c in df_final.columns if c != TARGET]

X = df_labeled[FEATURE_COLS]
y = df_labeled[TARGET]

# Split Train / Temp (70% - 30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
# Split Validation / Test (15% - 15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}")
print(f"Colonnes features ({len(FEATURE_COLS)}) :")
print(FEATURE_COLS)

import os
os.makedirs("../data/processed", exist_ok=True)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_val.to_csv("../data/processed/X_val.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_val.to_csv("../data/processed/y_val.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)
df_final.to_csv("../data/processed/dataset_clean.csv", index=False)
df_unlabeled.to_csv("../data/processed/dataset_unlabeled.csv", index=False)

# Full files for Personne 2 & 3 (includes Rapport_Collecte and Prix_Revente)
pd.concat([X_train, y_train], axis=1).to_csv("../data/processed/train.csv", index=False)
pd.concat([X_val,   y_val],   axis=1).to_csv("../data/processed/val.csv",   index=False)
pd.concat([X_test,  y_test],  axis=1).to_csv("../data/processed/test.csv",  index=False)

print(f"\nTailles des splits : Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")
print("Sauvegarde effectuée dans le dossier ../data/processed/")

Train : 6990 | Val : 1498 | Test : 1498
Colonnes features (13) :
['Poids', 'Volume', 'Conductivite', 'Opacite', 'Rigidite', 'Prix_Revente', 'Source_Centre_Tri', 'Source_Collecte_Citoyenne', 'Source_Inconnu', 'Source_Usine_A', 'Source_Usine_B', 'Source_encoded', 'Rapport_Collecte']

Tailles des splits : Train=6990, Val=1498, Test=1498
Sauvegarde effectuée dans le dossier ../data/processed/


In [19]:
splits_data = []
for split_name, split_y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    for cat, count in split_y.value_counts().items():
        splits_data.append({"Split": split_name, "Categorie": cat, "Count": count})

df_splits_viz = pd.DataFrame(splits_data)
fig_split = px.bar(
    df_splits_viz,
    x="Categorie", y="Count", color="Split", barmode="group",
    title="Distribution des classes par split (stratifiée)",
    color_discrete_sequence=["#3498db", "#e67e22", "#2ecc71"],
    height=400
)
fig_split.show()

## Conclusion
Au terme de ce notebook, les données ont été intégralement nettoyées. Le mécanisme de la donnée manquante a été exploré (Little's test) et imputé (IterativeImputer). Après encodage One-Hot de la source et standardisation numérique, nous avons isolé les observations labélisées et constitué un split équilibré (70% train / 15% validation / 15% test). L'ensemble du jeu de données est désormais prêt pour l'entraînement d'algorithmes de machine learning.
